In [1]:
import pandas as pd

In [2]:
hora = pd.read_csv("cr7_chats.csv")
hora.head()

,video_id,author,comment,likes,published_at,is_reply
0,JKEuUQXq9PY,@bishalghalan7146,😂😂😂,0,2026-03-24T18:13:28Z,False
1,JKEuUQXq9PY,@avinavbudhathoki8155,Dai 4 month jati wait garnu Tespaxi bala kinu ...,0,2026-03-24T17:42:25Z,False
2,JKEuUQXq9PY,@KALTIIEG-b2r,"<a href=""https://www.youtube.com/watch?v=JKEuU...",0,2026-03-24T17:32:15Z,False
3,JKEuUQXq9PY,@cr7horaaYT,Bhai license ko lagi agadi sikeko dekhenau 😂,1,2026-03-24T17:56:08Z,True
4,JKEuUQXq9PY,@mr.k.khadka1371,"Keta haru le pasina chuhaune, ghantey le moj g...",0,2026-03-24T16:43:57Z,False


In [4]:
hora_chats = hora['comment']

In [37]:
len(hora_chats)

49241

In [49]:
import re

def clean_youtube_chat(raw_lines, remove_all_duplicates=False, normalize_repeated_chars=False):
    """
    Cleans YouTube chat lines:
      - Removes line numbers, HTML tags, HTML entities, URLs, emojis
      - Removes words starting with @ (mentions)
      - Removes timestamps (e.g., 0:13, 0:14, 1:23:45, etc.)
      - Removes any leftover HTML fragments like <a href="
      - Removes ALL types of links (YouTube, http, https, short links, etc.)
      - Removes lines containing specific keywords (instagram, facebook, etc.)
      - Removes common promotional/repetitive phrases
      - Normalizes extra dots/punctuation
      - Optionally normalizes repeated characters
      - Removes messages that consist of 1 or 2 words after cleaning
      - Removes duplicate messages
    """
    cleaned_messages = []
    prev_msg = None
    seen = set()

    # Keywords to remove entire line (case-insensitive)
    keywords_to_remove = [
        r'instagram',
        r'facebook',
        r'twitter',
        r'tiktok',
        r'snapchat',
        r'telegram',
        r'discord',
        r'whatsapp',
        r'onlyfans',
        r'patreon',
        r'ko-fi',
        r'buff',
    ]

    # Common promotional/repetitive phrases to remove
    promotional_phrases = [
        r'video mann parey maa like',
        r'share ra subscribe garnu hola',
        r'also follow on instagram',
        r'like.*share.*subscribe',
        r'follow on instagram',
        r'click.*subscribe',
        r'press.*bell.*icon',
        r'don\'t forget to subscribe',
        r'please subscribe',
        r'like and share',
        r'subscribe to my channel',
        r'follow me on',
        r'check out my',
        r'support my channel',
    ]

    # Emoji removal pattern
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001F900-\U0001F9FF"
        "\U0001FA70-\U0001FAFF"
        "]+", flags=re.UNICODE
    )

    for line in raw_lines:
        # Convert to string if not already
        original_line = str(line)
        
        # --- Check if line contains any keyword to remove (case-insensitive) ---
        should_remove = False
        for keyword in keywords_to_remove:
            if re.search(keyword, original_line, re.IGNORECASE):
                should_remove = True
                break
        
        if should_remove:
            continue  # Skip this entire line
        
        # --- Basic cleaning ---
        line = re.sub(r'&quot;', '', original_line)
        line = re.sub(r'&amp;', '&', line)
        line = re.sub(r'&lt;', '<', line)
        line = re.sub(r'&gt;', '>', line)
        line = re.sub(r'&nbsp;', ' ', line)
        line = re.sub(r'&[a-zA-Z]+;', '', line)  # Remove any other named HTML entities
    
        line = re.sub(r'&[^;\s]+;', '', line)
        line = re.sub(r'&#39', '', line)
        line = emoji_pattern.sub(r'', line)
        line = re.sub(r'[^\w\s.,!?;:()\-\'"]+', '', line)  # Remove special characters except basic punctuation
        line = re.sub(r'\s+', ' ', line).strip()  # Normalize whitespace
        line = re.sub(r'\.{2,}', '', line)  # Remove extra dots
        line = re.sub(r'([!?])\1+', r'\1', line)  # Normalize repeated punctuation

        
        # --- Remove ALL types of links/URLs ---
        line = re.sub(r'https?://\S+', '', line)
        line = re.sub(r'ftp://\S+', '', line)
        line = re.sub(r'www\.\S+', '', line)
        line = re.sub(r'youtu\.be/\S+', '', line)
        line = re.sub(r'youtube\.com/\S+', '', line)
        line = re.sub(r'\b(?:https?|ftp|file)://\S+', '', line)
        line = re.sub(r'\S+\.(com|org|net|io|co|in|ly|be)/\S+', '', line)
        line = re.sub(r'http\S+', '', line)
        
        # --- Remove HTML tags and fragments ---
        line = re.sub(r'<[^>]+>', '', line)
        line = re.sub(r'</?[a-zA-Z][^>]*>?', '', line)
        line = re.sub(r'href\s*=\s*["\'][^"\']*["\']', '', line)
        line = re.sub(r'href\s*=\s*[^\s>]+', '', line)
        line = re.sub(r'<[^>]*>?', '', line)
        
        # --- Remove line numbers ---
        line = re.sub(r'^\s*\d+\s+', '', line)

        # --- Remove timestamps ---
        line = re.sub(r'\b\d+:\d+(?::\d+)?\b', '', line)
        line = re.sub(r'\d+:\d+(?::\d+)?\s*', '', line)
        
        # --- Remove words starting with @ ---
        line = re.sub(r'@\S+', '', line)
        line = re.sub(r'@', '', line)

        # --- Remove promotional/repetitive phrases ---
        for phrase in promotional_phrases:
            line = re.sub(phrase, '', line, flags=re.IGNORECASE)
        
        # Remove leftover "also" and "ra" if they become standalone
        line = re.sub(r'\balso\b', '', line, flags=re.IGNORECASE)
        line = re.sub(r'\bra\b', '', line, flags=re.IGNORECASE)

        # --- Clean extra dots and punctuation ---
        line = re.sub(r'\.{2,}', '', line)
        line = re.sub(r'([!?])\1+', r'\1', line)
        line = re.sub(r'^[.\s]+|[.\s]+$', '', line)

        # --- Optional: Normalize repeated characters ---
        if normalize_repeated_chars:
            line = re.sub(r'(.)\1{2,}', r'\1', line)

        # Normalize whitespace
        line = re.sub(r'\s+', ' ', line).strip()

        if not line:
            continue


        word_count = len(line.split())
        if word_count <= 5:
            continue

        # --- Remove duplicate messages ---
        if remove_all_duplicates:
            if line in seen:
                continue
            seen.add(line)
        else:
            if line == prev_msg:
                continue

        cleaned_messages.append(line)
        prev_msg = line

    return cleaned_messages

def save_chat_to_txt(cleaned_messages, filename="cleaned_chat.txt"):
    """
    Saves cleaned chat messages to a text file.
    Each message on a new line.
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for msg in cleaned_messages:
            f.write(msg + '\n')
    print(f"✅ Saved {len(cleaned_messages)} messages to '{filename}'")

sample_chat = hora_chats

cleaned_messages = clean_youtube_chat(sample_chat, remove_all_duplicates=False, normalize_repeated_chars=False)

# Save to file
save_chat_to_txt(cleaned_messages, "cleaned_chat_no_instagram.txt")

✅ Saved 14992 messages to 'cleaned_chat_no_instagram.txt'


In [50]:
samachar = pd.read_csv("samacharpati_chats.csv")
samachar.head()

,video_id,author,comment,likes,published_at,is_reply
0,tdDhpiIcexc,@KshetraDangi,हर्कले हावा तालमा कुरा गर्छन। जिम्मेवार व्यक्त...,0,2026-03-25T02:01:02Z,False
1,tdDhpiIcexc,@RamprasaddahalDahal,धन्यवाद म जनकपुर बाट।,0,2026-03-25T02:00:46Z,False
2,tdDhpiIcexc,@JayPun-fn3wr,Ragular,0,2026-03-25T01:54:31Z,False
3,tdDhpiIcexc,@KshetraDangi,नेपाली हु भन्नेले अरु भाषामा सपथ लिन पाइदैन 🎉,0,2026-03-25T01:54:24Z,False
4,tdDhpiIcexc,@JogBdrKUNWAR,Good morning Ramshran Sir Best Newes Center Re...,0,2026-03-25T01:53:23Z,False


In [53]:
samachar['comment'][4]

'Good morning Ramshran Sir Best Newes Center Regular From Nepalgunj. ❤❤❤'